# Rainfall Analysis and Planting Advisory for a Farmer-Support NGO

**Author:** Chala Wakshuma

A farmer-support NGO sends SMS advice on planting windows to farmers in five counties. It has monthly rainfall readings (mm) from local weather stations, but the Machakos station's data was typed in by hand and contains errors. This notebook cleans that data, analyses rainfall patterns with NumPy, saves a summary to `rainfall_summary.csv`, and ends with a short planting advisory.

**How to run:** Runtime > Restart session and run all (or `Kernel > Restart & Run All`). Requires only `numpy`; the `csv` module is part of Python.

## Dataset

The data is copied exactly as provided: four clean county rows and the raw Machakos row, which arrived as text. I also add a `months` array of month names (used later to turn positions into names) and a named constant for the dry-month threshold.

In [1]:
import csv

import numpy as np

counties = ["Nairobi", "Nakuru", "Kisumu", "Mombasa", "Machakos"]
clean_rows = [
    [64, 56, 92, 219, 176, 35, 18, 24, 31, 61, 165, 104],   # Nairobi
    [35, 42, 70, 145, 120, 78, 96, 105, 72, 55, 60, 38],    # Nakuru
    [58, 90, 150, 240, 180, 95, 70, 85, 90, 88, 130, 95],   # Kisumu
    [30, 15, 55, 150, 240, 110, 90, 65, 60, 90, 100, 70],   # Mombasa
]
machakos_raw = ["48", "35", "88", "N/A", "140", " 20 ",
                "9", "12", "", "45", "182", "96"]

months = np.array(["Jan", "Feb", "Mar", "Apr", "May", "Jun",
                   "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"])
DRY_THRESHOLD_MM = 50   # a month below this is a "dry month"

## Task 1: Clean the Machakos row

**Approach:** Two small functions. `clean_rainfall_value()` tries to turn one text value into an integer with `int()` inside `try/except ValueError`, and returns `None` when it cannot. `int()` already ignores surrounding spaces, so `" 20 "` becomes `20`, while `"N/A"` and `""` raise `ValueError`. I also treat a negative number as invalid, because rainfall cannot be below zero. `clean_machakos_readings()` then averages the valid values, rounds that average, and uses it to fill every invalid month, and returns which months were filled.

In [2]:
def clean_rainfall_value(text):
    """Convert one raw rainfall reading to an integer, or return None if it is invalid.

    Surrounding spaces are accepted (" 20 " -> 20). Text that is not a whole number
    ("N/A", "") and negative numbers are treated as invalid.
    """
    try:
        value = int(text)
        if value < 0:
            raise ValueError("rainfall cannot be negative")
        return value
    except ValueError:
        return None


def clean_machakos_readings(raw_values):
    """Clean a list of raw readings and fill invalid ones with the rounded mean of the valid ones.

    Returns (cleaned_values, imputed_positions, fill_value).
    """
    parsed = [clean_rainfall_value(item) for item in raw_values]
    valid = [value for value in parsed if value is not None]
    if not valid:
        raise ValueError("no valid readings to calculate an average from")
    fill_value = round(sum(valid) / len(valid))
    imputed_positions = [i for i, value in enumerate(parsed) if value is None]
    cleaned = [fill_value if value is None else value for value in parsed]
    return cleaned, imputed_positions, fill_value


machakos_clean, imputed_positions, fill_value = clean_machakos_readings(machakos_raw)
valid_count = len(machakos_raw) - len(imputed_positions)

print(f"Valid readings: {valid_count} of {len(machakos_raw)}")
print(f"Rounded average of the valid readings: {fill_value} mm")
print("Imputed months:", ", ".join(months[imputed_positions]))
print("Cleaned Machakos row:", machakos_clean)

Valid readings: 10 of 12
Rounded average of the valid readings: 68 mm
Imputed months: Apr, Sep
Cleaned Machakos row: [48, 35, 88, 68, 140, 20, 9, 12, 68, 45, 182, 96]


## Task 2: Combine into one 5 × 12 NumPy array

**Approach:** Stack the four clean rows and the cleaned Machakos row into one array (rows are counties, columns are months), then confirm its shape and data type.

In [3]:
rainfall = np.array(clean_rows + [machakos_clean])

print("Shape:", rainfall.shape)
print("Dtype:", rainfall.dtype)
print()
print(rainfall)
assert rainfall.shape == (5, 12)

Shape: (5, 12)
Dtype: int64

[[ 64  56  92 219 176  35  18  24  31  61 165 104]
 [ 35  42  70 145 120  78  96 105  72  55  60  38]
 [ 58  90 150 240 180  95  70  85  90  88 130  95]
 [ 30  15  55 150 240 110  90  65  60  90 100  70]
 [ 48  35  88  68 140  20   9  12  68  45 182  96]]


## Task 3: Annual totals and monthly averages

**Approach:** `sum(axis=1)` adds across the months of each county (one total per row), and `mean(axis=0)` averages down the counties for each month (one average per column).

In [4]:
annual_totals = rainfall.sum(axis=1)
monthly_average = rainfall.mean(axis=0)

print("Annual total rainfall per county (mm):")
for county, total in zip(counties, annual_totals):
    print(f"  {county:<9} {total:>5,}")

print("\nAverage rainfall per month across all counties (mm):")
for month, average in zip(months, monthly_average):
    print(f"  {month}  {average:>6.1f}")

Annual total rainfall per county (mm):
  Nairobi   1,045
  Nakuru      916
  Kisumu    1,371
  Mombasa   1,075
  Machakos    811

Average rainfall per month across all counties (mm):
  Jan    47.0
  Feb    47.6
  Mar    91.0
  Apr   164.4
  May   171.2
  Jun    67.6
  Jul    56.6
  Aug    58.2
  Sep    64.2
  Oct    67.8
  Nov   127.4
  Dec    80.6


## Task 4: Wettest month per county

**Approach:** `np.argmax(axis=1)` returns the column position of each row's maximum. Indexing the `months` array with those positions turns them into month names, and `zip()` pairs each county with its result.

In [5]:
wettest_idx = np.argmax(rainfall, axis=1)
wettest_months = months[wettest_idx]
wettest_mm = rainfall.max(axis=1)

for county, month, mm in zip(counties, wettest_months, wettest_mm):
    print(f"{county:<9} wettest month: {month} ({mm} mm)")

Nairobi   wettest month: Apr (219 mm)
Nakuru    wettest month: Apr (145 mm)
Kisumu    wettest month: Apr (240 mm)
Mombasa   wettest month: May (240 mm)
Machakos  wettest month: Nov (182 mm)


## Task 5: Dry months (below 50 mm)

**Approach:** Comparing the whole array with the threshold gives a boolean mask (`True` where a month is dry). Summing a boolean mask along `axis=1` counts the `True` values in each row.

In [6]:
dry_mask = rainfall < DRY_THRESHOLD_MM
dry_counts = dry_mask.sum(axis=1)

for county, count in zip(counties, dry_counts):
    print(f"{county:<9} dry months: {count}")

Nairobi   dry months: 4
Nakuru    dry months: 3
Kisumu    dry months: 0
Mombasa   dry months: 2
Machakos  dry months: 6


## Task 6: Planting window (two consecutive months with the most rain)

**Approach:** Adding the array to a copy of itself shifted by one month, `R[:, :-1] + R[:, 1:]`, gives every consecutive two-month total in one step (11 pairs per county: Jan-Feb, Feb-Mar, and so on). `argmax(axis=1)` picks the best pair, and `np.char.add` builds labels like `Apr-May`. Windows are tested within the calendar year only (a Dec-Jan pair would need the next year's data).

In [7]:
pair_sums = rainfall[:, :-1] + rainfall[:, 1:]          # shape (5, 11)
best_pair_idx = np.argmax(pair_sums, axis=1)
best_pair_mm = pair_sums.max(axis=1)

planting_windows = np.char.add(np.char.add(months[best_pair_idx], "-"), months[best_pair_idx + 1])

for county, window, mm in zip(counties, planting_windows, best_pair_mm):
    print(f"{county:<9} planting window: {window} ({mm} mm combined)")

Nairobi   planting window: Apr-May (395 mm combined)
Nakuru    planting window: Apr-May (265 mm combined)
Kisumu    planting window: Apr-May (420 mm combined)
Mombasa   planting window: Apr-May (390 mm combined)
Machakos  planting window: Nov-Dec (278 mm combined)


## Task 7: Save `rainfall_summary.csv`

**Approach:** Build one row per county with the five required columns and write it with the `csv` module (`.tolist()` converts NumPy values to plain Python values first). The file is then read back to confirm it has exactly five data rows.

In [8]:
header = ["county", "annual_total_mm", "wettest_month", "dry_months", "planting_window"]
summary_rows = list(zip(counties,
                        annual_totals.tolist(),
                        wettest_months.tolist(),
                        dry_counts.tolist(),
                        planting_windows.tolist()))

with open("rainfall_summary.csv", "w", newline="") as file:
    writer = csv.writer(file, lineterminator="\n")
    writer.writerow(header)
    writer.writerows(summary_rows)

# read the file back and check it
with open("rainfall_summary.csv", newline="") as file:
    saved = list(csv.reader(file))

assert saved[0] == header and len(saved) == 6            # header + exactly five counties
for line in saved:
    print(", ".join(line))

county, annual_total_mm, wettest_month, dry_months, planting_window
Nairobi, 1045, Apr, 4, Apr-May
Nakuru, 916, Apr, 3, Apr-May
Kisumu, 1371, Apr, 0, Apr-May
Mombasa, 1075, May, 2, Apr-May
Machakos, 811, Nov, 6, Nov-Dec


## Task 8: Write the advisory

**Approach:** The advisory below uses the numbers from Tasks 1 to 7 and includes the data-quality caveat from the Machakos cleaning. The next section lists the data-quality notes in full.

## Planting Advisory

For Nairobi, Nakuru, Kisumu and Mombasa, the best planting window is **April–May**, the two months with the most combined rainfall (395, 265, 420 and 390 mm respectively), so SMS advice should go out about a month before, in March. Kisumu is the wettest and most reliable county (1,371 mm a year with no dry months), while Nakuru receives the least of these four (916 mm). Machakos is different: its best window is **November–December** (278 mm), so its advice should be sent in October and not copied from the other counties. Machakos also has the most dry months (6 of 12 below 50 mm), so planting advice there should come with drought-tolerant crop and water-saving guidance. **Caveat:** two Machakos readings (April and September) were unreadable and were estimated at 68 mm, so treat the Machakos advice as provisional until the station's paper records are checked. All of this rests on a single year of monthly data, so the windows should be confirmed against several years before they are treated as firm.

## Data Quality Notes

- **Machakos was cleaned, not measured.** Two of its twelve readings were invalid (`"N/A"` for April and an empty string for September) and were replaced with 68 mm, the rounded average of the ten valid readings (67.5, which Python rounds to 68). The reading `" 20 "` had stray spaces but was a valid number, so it was kept as 20.
- **The imputed months are a meaningful share of the total.** The two estimates add 136 mm to Machakos's 811 mm annual total (about 17%).
- **A flat average ignores seasonality.** April is a wet month for the other four counties, so 68 mm may understate Machakos's true April rainfall.
- **The advice is robust to that, but the dry-month count is not.** The November–December window stays the best unless the true April reading exceeds 138 mm, whereas the Machakos dry-month count could really be anywhere from 6 to 8 depending on the true April and September values (see the check below).
- **The other four counties were used as supplied.** The readings are plausible (see the range check), but typing errors that still look like normal rainfall cannot be detected.
- **Limits of the data.** It is one year, one station per county, and monthly totals only, so it says nothing about when rains start, dry spells inside a month, or year-to-year variation.

In [9]:
# Range check: monthly rainfall readings should be non-negative and within a plausible range
print(f"Smallest reading: {rainfall.min()} mm, largest reading: {rainfall.max()} mm")
assert rainfall.min() >= 0 and rainfall.max() < 500

# How much of the Machakos total is estimated?
machakos = rainfall[4]
imputed_total = fill_value * len(imputed_positions)
print(f"Imputed share of Machakos total: {imputed_total} of {annual_totals[4]} mm ({imputed_total / annual_totals[4]:.1%})")

# How wrong would the April estimate have to be to change the Machakos advice?
nov_dec = pair_sums[4, 10]
april_to_beat = nov_dec - machakos[4]          # April + May (140 mm) must exceed Nov-Dec to win
print(f"Machakos Nov-Dec = {nov_dec} mm; April-May only wins if April were above {april_to_beat} mm (estimate used: {machakos[3]} mm)")

# Dry-month count if the two estimated months were actually dry or not
low, high = dry_counts[4], dry_counts[4] + len(imputed_positions)
print(f"Machakos dry months: {low} as calculated, up to {high} if both imputed months were really below {DRY_THRESHOLD_MM} mm")

Smallest reading: 9 mm, largest reading: 240 mm
Imputed share of Machakos total: 136 of 811 mm (16.8%)
Machakos Nov-Dec = 278 mm; April-May only wins if April were above 138 mm (estimate used: 68 mm)
Machakos dry months: 6 as calculated, up to 8 if both imputed months were really below 50 mm
